In [4]:
#Initialize Variables
import yfinance as yf
import pandas as pd
from scipy import stats
import numpy as np
import xml.etree.ElementTree as ET
import requests
from datetime import date
from datetime import datetime
from scipy.stats import norm
from openpyxl import load_workbook
import os

#Pull SEC Tickers
URLtwo = "https://www.sec.gov/files/company_tickers.json"
headers={"User-Agent":"Unemployed tylernorman1212@gmail.com","Accept-Encoding": "gzip,deflate","Host":"www.sec.gov" }
SEC_tickers = requests.get(url = URLtwo, headers = headers)

if SEC_tickers.status_code == 200:
    sec_tickdat = SEC_tickers.json()
    total_tickers =  [entry['ticker'] for entry in sec_tickdat.values()]
else:
    print(f"Failed to fetch data. HTTP Status Code: {SEC_tickers.status_code}")

#Pull Rf
fed_scrape ='https://home.treasury.gov/sites/default/files/interest-rates/yield.xml'
response = requests.get(fed_scrape)  
if response.status_code == 200:
    treas_xml = response.content
    root = ET.fromstring(treas_xml)
    subchildren = root.findall('.//AVERAGE_1MONTH')
    if subchildren:
        latest_subchild = subchildren[-1]
    else:
        print('unable to locate published average.')
        subchildren_1 = root.findall('.//BC_1MONTH')
        if subchildren_1:
            lastest_subchild = subchildren[-1]
        else:
            print('unable to find published data')
            latest_subchild = str(input('Enter manually'))
else:
    print(f"failed to retrieve treasury data. Status code: {response.status_code}")

rf = float(latest_subchild.text)/100

#Realistic Info
rb = float(input('input borrowing rate'))

IKBR_fee = .65
ORF = .02875
FINRA = .0035
SEC_fee = .0000278
FINRA_fee = .00279
OCC_fee = .025

total_fees = IKBR_fee+ORF+FINRA+SEC_fee+FINRA_fee+OCC_fee

#Pull VIX and assign Volatility
vix = yf.Ticker("^VIX")
vix_rtd = vix.history(period = '1d', interval = '1m')['Close'].iloc[-1]
sigma = vix_rtd/100


#Get Options Matrix, assign time, strike and spot price

for tick in total_tickers:

    Time = []
    Time_p = []
    Strike = []
    Strike_p = []
    Date = []
    Date_p = []
    Spot = []
    Spot_p = []
    Volat = []
    ask_c = []
    ask_p = []
    rfr = []
    bid_c = []
    bid_p = []
    volume_c = []
    volume_p = []

#Assign Spot
    s = yf.Ticker(tick).fast_info['lastPrice']
    opt_date = yf.Ticker(tick).options
    if not opt_date:
        print(f"no options for: {tick}")
        continue

    for i in opt_date:
        options = yf.Ticker(tick).option_chain(i)
        call_strikes = options.calls['strike']
    
        #Get Time
        exp_1 = datetime.strptime(i,"%Y-%m-%d").date()
        exp_2 = date.today()
        tenor = exp_1 - exp_2
    
        call_price = options.calls['ask']
        put_price = options.puts['ask']
    
        call_sell = options.calls['bid']
        put_sell = options.puts['bid']
        call_volume = options.calls['volume']
        put_volume = options.puts['volume']
    
        for prc in call_price:
            ask_c.append(prc)   
        for prp in put_price:
            ask_p.append(prp)
        
        for scall in call_sell:
            bid_c.append(scall)
        for sput in put_sell:
            bid_p.append(sput)
        
        for puv in put_volume:
            volume_p.append(puv)
        for cuv in call_volume:
            volume_c.append(cuv)

        put_strikes = options.puts['strike']
        for ps in put_strikes:
            Time_p.append(tenor.days/365)
            Strike_p.append(ps)
            Date_p.append(i)
            Spot_p.append(s)

        #Get Exercise
        for c in call_strikes:
            Time.append(tenor.days/365)
            Strike.append(c)
            Date.append(i)
            Spot.append(s)
            Volat.append(sigma)
            rfr.append(rf)
    
        
    df1 = pd.DataFrame(Time, index = Date, columns = ['Tenor'])
    df2 = pd.DataFrame(Strike,index = Date, columns = ['Strike'])
    df3 = pd.DataFrame(Spot,index = Date, columns = ['Spot'])
    df4 = pd.DataFrame(Volat,index = Date, columns = ['Volatility'])
    df5 = pd.DataFrame(ask_c,index = Date, columns = ['ask call'])
    df6 = pd.DataFrame(rfr,index = Date, columns = ['rf'])
    df7 = pd.DataFrame(bid_c,index = Date, columns = ['bid call'])
    df8 = pd.DataFrame(volume_c,index = Date, columns = ['volume_c'])

    scholes_df = pd.concat([df1,df2],axis=1)
    black_df = pd.concat([scholes_df, df3],axis=1)
    blsh_df = pd.concat([black_df,df4],axis=1)
    blshl_df = pd.concat([blsh_df,df5],axis=1)
    blshlm_df = pd.concat([blshl_df,df7],axis=1)
    blss_df = pd.concat([blshlm_df,df8],axis=1)
    bls_df = pd.concat([blss_df,df6],axis=1)


    bls_df['d1'] = (np.log(bls_df['Spot']/bls_df['Strike']) + (rf + 0.5 * sigma**2)*bls_df['Tenor'])/(sigma*np.sqrt(bls_df['Tenor'])) 
    bls_df['d2'] = bls_df['d1'] - sigma*np.sqrt(bls_df['Tenor'])
    bls_df['d1'] = pd.to_numeric(bls_df['d1'], errors='coerce')
    bls_df['d2'] = pd.to_numeric(bls_df['d2'], errors='coerce')

    
    bls_df['BlSchCP'] = bls_df['Spot']*norm.cdf(bls_df['d1']) - bls_df['Strike']*np.exp(rf*-1*bls_df['Tenor'])*norm.cdf(bls_df['d2'])
    bls_df['ask call - blsh'] = bls_df['ask call'] - bls_df['BlSchCP']

    df1p = pd.DataFrame(Time_p, index = Date_p, columns = ['Tenor'])
    df2p = pd.DataFrame(Strike_p, index = Date_p, columns = ['Strike'])
    df3p = pd.DataFrame(Spot_p, index = Date_p, columns = ['Spot'])
    df4p=  pd.DataFrame(bid_p, index = Date_p, columns = ['bid put'])
    df5p = pd.DataFrame(ask_p, index = Date_p, columns = ['ask put'])
    df6p = pd.DataFrame(volume_p, index = Date_p, columns = ['volume_p'])

    scholes_dfp = pd.concat([df1p,df2p],axis=1)
    black_dfp = pd.concat([scholes_dfp, df3p],axis=1)
    blsh_dfp = pd.concat([black_dfp,df5p],axis=1)
    bbl_dfp = pd.concat([blsh_dfp,df6p],axis=1)
    bls_dfp = pd.concat([bbl_dfp,df4p],axis=1)

    bls_dfp['d1p'] = (np.log(bls_dfp['Spot']/bls_dfp['Strike']) + (rf +0.5*sigma**2)*bls_dfp['Tenor'])/(sigma*np.sqrt(bls_dfp['Tenor'])) 
    bls_dfp['d2p'] = bls_dfp['d1p'] - sigma*np.sqrt(bls_dfp['Tenor'])
    bls_dfp['d1p'] = pd.to_numeric(bls_dfp['d1p'], errors='coerce')
    bls_dfp['d2p'] = pd.to_numeric(bls_dfp['d2p'], errors='coerce')
    
    
    bls_dfp['BlSchPP'] = bls_dfp['Strike']*np.exp(rf*-1*bls_dfp['Tenor'])*norm.cdf(-1*bls_dfp['d2p']) - bls_dfp['Spot']*norm.cdf(-1*bls_dfp['d1p'])
    bls_dfp['ask put - blsh'] = bls_dfp['ask put'] - bls_dfp['BlSchPP'] 

    #put-call-parity
    bl_df = pd.merge(bls_df,bls_dfp, on = ['Tenor','Strike'], how = 'inner')

    def check_pcp(row, rf, rb):
        lhs = row['ask call']
        rhs = row['Spot_x'] - row['Strike'] * np.exp(-rf * row['Tenor']) + row['bid put']
    
        if np.isclose(lhs, rhs, atol=1e-6):
            return pd.Series({
                'pcp holds?': True,
                'arb call': 0,
                'arb short(+),long(-) stock': 0,
                'arb put': 0,
                'arb profit': 0,
            })

        # Case 1: Call is too expensive → sell call, buy stock, buy put
        elif lhs > rhs:
        
            net_outflow = (row['Spot_x'] + row['ask put'] - row['bid call'])
            repay_at_expiry = net_outflow * np.exp(rb * row['Tenor'])
            arb_profit = row['Strike'] - repay_at_expiry
        
            return pd.Series({
                'pcp holds?': False,
                'arb call': row['bid call'],  # sell call → +cash
                'arb short(+),long(-) stock': -row['Spot_x'],  # buy stock → -cash
                'arb put': -row['ask put'],  # buy put → -cash
                'arb profit': arb_profit - total_fees
            })

    # Case 2: Put is too expensive → sell put, short stock, buy call
        else:
            # Now check the put-based parity expression:
            lhs_put = row['ask put']
            rhs_put = row['bid call'] - row['Spot_x'] + row['Strike'] * np.exp(-rf * row['Tenor'])
        
            if lhs_put > rhs_put:
            
                net_inflow = row['bid put'] + row['Spot_x'] - row['ask call']
                future_value = net_inflow * np.exp(rf * row['Tenor'])
                arb_profit = future_value - row['Strike']
            
                return pd.Series({
                    'pcp holds?': False,
                    'arb call': -row['ask call'],  # buy call → -cash
                    'arb short(+),long(-) stock': row['Spot_x'],  # short stock → +cash
                    'arb put': row['bid put'], # sell put → +cash
                    'arb profit': arb_profit - total_fees
                })
            else:
                return pd.Series({
                    'pcp holds?': np.nan,  # no clear arbitrage
                    'arb call': 0,
                    'arb short(+),long(-) stock': 0,
                    'arb put': 0,
                    'arb profit': 0
                })

# Apply to each row
    bl_df[['pcp holds?', 'arb call', 'arb short(+),long(-) stock', 'arb put', 'arb profit']] = bl_df.apply(
        lambda row: check_pcp(row, rf, rb), axis=1
    )
    bl_df['Annualized_ROR'] = (1+(bl_df['arb profit']/((bl_df['arb call']+bl_df['arb put']+bl_df['arb short(+),long(-) stock'])*np.exp(rf*bl_df['Tenor'])))**(1/bl_df['Tenor']))-1
    bl_df = bl_df.drop(['Spot_y','d1p','d2p'], axis=1)
    bl_df = bl_df[bl_df['arb profit'] > 0.0] 
    bl_df = bl_df[np.isfinite(bl_df['Annualized_ROR'])]
    #bl_df = bl_df[bl_df['volume_p']>10]
    #bl_df = bl_df[bl_df['volume_c']>10]
    bl_df = bl_df[bl_df['ask call']+bl_df['Strike'] > bl_df['Spot_x']]
    bl_df.insert(0, 'Ticker', tick)

   
    excel_path = "options.xlsx"
    sheet_name = "ArbitrageResults"

# Create a new file if it doesn't exist
    if not os.path.exists(excel_path):
    # Write with headers
        bl_df.to_excel(excel_path, sheet_name=sheet_name, index=False)
    else:
    # Load the existing workbook
        book = load_workbook(excel_path)

    # Get the next empty row in the target sheet
        if sheet_name in book.sheetnames:
            sheet = book[sheet_name]
            next_row = sheet.max_row
        else:
            next_row = 0  # If sheet doesn't exist yet

    # Use ExcelWriter and properly set the sheets dict
        with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
            bl_df.to_excel(writer, sheet_name=sheet_name, startrow=next_row, index=False, header=False if next_row > 0 else True)


    print(f"{tick} data added to {sheet_name} sheet in {excel_path}")


input borrowing rate .28


AAPL data added to ArbitrageResults sheet in options.xlsx
MSFT data added to ArbitrageResults sheet in options.xlsx
NVDA data added to ArbitrageResults sheet in options.xlsx
AMZN data added to ArbitrageResults sheet in options.xlsx
GOOGL data added to ArbitrageResults sheet in options.xlsx
META data added to ArbitrageResults sheet in options.xlsx
BRK-B data added to ArbitrageResults sheet in options.xlsx
TSLA data added to ArbitrageResults sheet in options.xlsx
TSM data added to ArbitrageResults sheet in options.xlsx
AVGO data added to ArbitrageResults sheet in options.xlsx
WMT data added to ArbitrageResults sheet in options.xlsx
LLY data added to ArbitrageResults sheet in options.xlsx
V data added to ArbitrageResults sheet in options.xlsx
JPM data added to ArbitrageResults sheet in options.xlsx
UNH data added to ArbitrageResults sheet in options.xlsx
SPY data added to ArbitrageResults sheet in options.xlsx
XOM data added to ArbitrageResults sheet in options.xlsx
MA data added to Arbit

$CHA: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
$CHA: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


KeyError: 'currentTradingPeriod'

In [18]:
#Pull SEC Tickers
URLtwo = "https://www.sec.gov/files/company_tickers.json"
headers={"User-Agent":"Unemployed tylernorman1212@gmail.com","Accept-Encoding": "gzip,deflate","Host":"www.sec.gov" }
SEC_tickers = requests.get(url = URLtwo, headers = headers)

if SEC_tickers.status_code == 200:
    sec_tickdat = SEC_tickers.json()
    total_tickers =  [entry['ticker'] for entry in sec_tickdat.values()]
else:
    print(f"Failed to fetch data. HTTP Status Code: {SEC_tickers.status_code}")
total_tickers

['AAPL',
 'MSFT',
 'NVDA',
 'AMZN',
 'GOOGL',
 'META',
 'BRK-B',
 'TSLA',
 'TSM',
 'AVGO',
 'WMT',
 'LLY',
 'V',
 'JPM',
 'UNH',
 'SPY',
 'XOM',
 'MA',
 'COST',
 'PG',
 'JNJ',
 'NFLX',
 'ORCL',
 'HD',
 'ABBV',
 'KO',
 'SAP',
 'TMUS',
 'RCIT',
 'NVO',
 'BABA',
 'BAC',
 'ASML',
 'CVX',
 'PM',
 'CRM',
 'CSCO',
 'MCD',
 'ABT',
 'IBM',
 'AZN',
 'NVS',
 'TM',
 'LIN',
 'MRK',
 'PEP',
 'WFC',
 'SHEL',
 'T',
 'HSBC',
 'VZ',
 'ACN',
 'GE',
 'PLTR',
 'FMX',
 'QQQ',
 'TMO',
 'HDB',
 'AXP',
 'ISRG',
 'MS',
 'AMGN',
 'RTX',
 'INTU',
 'RY',
 'BX',
 'IDEXY',
 'PGR',
 'DIS',
 'UL',
 'ADBE',
 'NOW',
 'GS',
 'QCOM',
 'PDD',
 'BKNG',
 'AMD',
 'SPGI',
 'TXN',
 'NEE',
 'CAT',
 'TJX',
 'GILD',
 'UBER',
 'SONY',
 'BSX',
 'SYK',
 'TTE',
 'DHR',
 'PFE',
 'MUFG',
 'UNP',
 'BLK',
 'CMCSA',
 'SNY',
 'EADSY',
 'LOW',
 'HON',
 'SCHW',
 'VRTX',
 'ADP',
 'BUD',
 'DE',
 'MMC',
 'RTNTF',
 'CB',
 'FI',
 'BMY',
 'C',
 'AIQUY',
 'COP',
 'IBN',
 'MDT',
 'BHP',
 'AMT',
 'AMAT',
 'BA',
 'PANW',
 'SPOT',
 'LMT',
 'TD',
 'ELV',

In [3]:
options = yf.Ticker('RCIT').options
if not options:
    print('poop')

poop
